In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = '/content/Dataset-CNBCI-Sentimented.csv'
SAVE_DIR = '/content/drive/MyDrive/NLP_Final_Project_Final/'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(f'{SAVE_DIR}/data', exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("Shape awal:", df.shape)
print("Kolom:", df.columns.tolist())
display(df.head())

required_cols = ['judul', 'sentimen']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Kolom '{col}' tidak ditemukan di dataset.")

df = df.dropna(subset=['judul', 'sentimen']).copy()

df = df.drop_duplicates(subset=['judul', 'sentimen']).copy()

print("Shape setelah cleaning dasar:", df.shape)

def normalize_financial_text(text):
    text = str(text).lower()
    text = re.sub(r'us\$', ' dolar ', text)
    text = re.sub(r'rp\s*', ' rupiah ', text)
    text = re.sub(r'%', ' persen ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_clean'] = df['judul'].apply(normalize_financial_text)

label_mapping = {
    'negatif': 0,
    'netral': 1,
    'positif': 2
}

df['label_id'] = df['sentimen'].map(label_mapping)

df = df.dropna(subset=['label_id']).copy()
df['label_id'] = df['label_id'].astype(int)

print("\nDistribusi label:")
print(df['sentimen'].value_counts())

print("\nDistribusi label_id:")
print(df['label_id'].value_counts().sort_index())

df_final = df[['judul', 'text_clean', 'sentimen', 'label_id']].copy()
df_final.to_csv(f'{SAVE_DIR}/data/clean_dataset.csv', index=False)

df_train, df_temp = train_test_split(
    df_final,
    test_size=0.20,
    stratify=df_final['label_id'],
    random_state=42
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    stratify=df_temp['label_id'],
    random_state=42
)

print("\nUkuran split:")
print("Train:", df_train.shape)
print("Val  :", df_val.shape)
print("Test :", df_test.shape)

print("\nDistribusi train:")
print(df_train['sentimen'].value_counts(normalize=True).round(4))

print("\nDistribusi val:")
print(df_val['sentimen'].value_counts(normalize=True).round(4))

print("\nDistribusi test:")
print(df_test['sentimen'].value_counts(normalize=True).round(4))

SPLIT_DIR = f'{SAVE_DIR}/data/splits'
os.makedirs(SPLIT_DIR, exist_ok=True)

df_train.to_csv(f'{SPLIT_DIR}/train_data.csv', index=False)
df_val.to_csv(f'{SPLIT_DIR}/val_data.csv', index=False)
df_test.to_csv(f'{SPLIT_DIR}/test_data.csv', index=False)

print("\nSemua file berhasil disimpan ke:")
print(SPLIT_DIR)

sample_preview = df_final[['judul', 'text_clean', 'sentimen']].sample(10, random_state=42)
display(sample_preview)

Mounted at /content/drive
Shape awal: (9819, 3)
Kolom: ['judul', 'tanggal', 'sentimen']


,judul,tanggal,sentimen
0,"Direktur Garuda Salman El Farisiy Meninggal, I...",2024/01/01,negatif
1,"Catatan Sejarah 2023, Indonesia Luncurkan Burs...",2024/01/01,netral
2,"Ramalan Bitcoin Paling Gila 2024, Naik 1.000% ...",2024/01/01,netral
3,Wah! Segini Harta Mantan Istri Prabowo Titiek ...,2024/01/01,netral
4,"Pensiun di Oktober 2024, Jokowi Kantongi Uang ...",2024/01/01,netral


Shape setelah cleaning dasar: (9747, 3)

Distribusi label:
sentimen
netral     4301
positif    2887
negatif    2559
Name: count, dtype: int64

Distribusi label_id:
label_id
0    2559
1    4301
2    2887
Name: count, dtype: int64

Ukuran split:
Train: (7797, 4)
Val  : (975, 4)
Test : (975, 4)

Distribusi train:
sentimen
netral     0.4413
positif    0.2961
negatif    0.2625
Name: proportion, dtype: float64

Distribusi val:
sentimen
netral     0.4410
positif    0.2964
negatif    0.2626
Name: proportion, dtype: float64

Distribusi test:
sentimen
netral     0.4410
positif    0.2964
negatif    0.2626
Name: proportion, dtype: float64

Semua file berhasil disimpan ke:
/content/drive/MyDrive/NLP_Final_Project_Final//data/splits


,judul,text_clean,sentimen
4553,Bank Regional Pakai Cara Ini Buat Maksimalkan ...,bank regional pakai cara ini buat maksimalkan ...,positif
7829,"Istri Ceraikan Suami Usai Cuan Rp 27 Miliar, T...",istri ceraikan suami usai cuan rupiah 27 milia...,negatif
1613,"Update Daftar Tersangka Kasus Timah, Sudah Tem...","update daftar tersangka kasus timah, sudah tem...",negatif
9634,"Gokil! Laba Bersih MDIY Meroket Hingga 205,6% ...","gokil! laba bersih mdiy meroket hingga 205,6 p...",positif
5553,"Dana Asing Kabur Bikin Rupiah Keok, Dolar AS N...","dana asing kabur bikin rupiah keok, dolar as n...",negatif
2674,"OJK Persempit Ruang Gerak Judi Online, Bikin U...","ojk persempit ruang gerak judi online, bikin u...",netral
4144,Top! Mandiri Investasi Raih Most Excellent Sha...,top! mandiri investasi raih most excellent sha...,positif
1600,Anak Buah Erick Thohir Tegaskan BUMN Butuh Duk...,anak buah erick thohir tegaskan bumn butuh duk...,netral
8416,"Tok! BI Rate Diputuskan Tetap 5,75%","tok! bi rate diputuskan tetap 5,75 persen",netral
588,"MIND ID Kuasai 34% Saham Vale, Erick: Ini Mome...","mind id kuasai 34 persen saham vale, erick: in...",netral
